[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/87_clip_loss_pdd260702_solution.ipynb)

# Solution: CLIP Contrastive Loss

Reference solution.

## 解析

**结论：CLIP 用对称的对比交叉熵。相似度矩阵 `logits = image @ text.T / tau`，第 `i` 张图与第 `i` 段文本互为正样本（对角线为标签），损失 = 图→文 与 文→图 两个方向的行 softmax 交叉熵的平均。**

### 构造
嵌入已 L2 归一化，`image @ text.T` 即成对余弦相似度，除以温度 `tau` 放大/缩小 logits 分布。`N` 个样本构成 `N x N` 矩阵，正确配对落在对角线，故 `labels = arange(N)`。

### 双向交叉熵
- 图→文：把每一行当作 `N` 类分类，正确类是对角元，`F.cross_entropy(logits, labels)`。
- 文→图：转置后同理，`F.cross_entropy(logits.T, labels)`。

取两者平均即对称损失。`F.cross_entropy` 内部已做数值稳定的 `log_softmax`（隐式减去行最大值），无需手动处理。

### 温度的作用
`tau` 越小，logits 越尖锐，正确配对被拉得越开，完美对齐时损失趋于 0；`tau` 越大分布越平缓，损失越接近 `ln N`。

### 复杂度
矩阵乘 `O(N^2 D)`，softmax/CE `O(N^2)`。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

def clip_loss(image_embeds, text_embeds, temperature):
    logits = image_embeds @ text_embeds.t() / temperature   # (N, N) similarity
    n = logits.size(0)
    labels = torch.arange(n, device=logits.device)          # image i <-> text i
    loss_i2t = F.cross_entropy(logits, labels)              # rows: image -> text
    loss_t2i = F.cross_entropy(logits.t(), labels)          # cols: text -> image
    return (loss_i2t + loss_t2i) / 2

In [ ]:
# Demo
import torch
img = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
txt = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
print(round(float(clip_loss(img, txt, 1.0)), 6))   # 0.313262

In [ ]:
from torch_judge import check
check('clip_loss_pdd260702')